# Diabetes Prediction

Standalone learning notebook for the first required application.

# Assignment 02: Three Intelligent Applications

This notebook implements the **learning stage** for the three required applications:

1. Diabetes classification using `diabetes.csv` and the tutorial workflow.
2. Vietnam house-price regression using `vietnam_housing_dataset.csv`.
3. Customer-support satisfaction/interest classification using `Customer_support_data.csv`.

Each application follows: **inspect -> clean -> represent -> split -> train -> evaluate -> persist -> inference test**.
All preprocessing is fitted inside a scikit-learn `Pipeline` to prevent data leakage.

## 0. Imports and reproducibility

In [6]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 30)
print('Working directory:', os.getcwd())

Working directory: e:\jupyter project


In [7]:
# 1. Application 1 - Diabetes prediction
# Problem: predict Outcome (0 = non-diabetic, 1 = diabetic) from eight clinical measurements.
# Tutorial data-quality rule: physiologically impossible zeros are treated as missing values.
diabetes = pd.read_csv('diabetes.csv')
print('Shape:', diabetes.shape)
display(diabetes.head())
print('\nMissing values:')
display(diabetes.isna().sum().to_frame('count').T)
print('\nDuplicate rows:', diabetes.duplicated().sum())
print('\nTarget distribution:')
display(diabetes['Outcome'].value_counts().rename_axis('Outcome').to_frame('count'))

DIABETES_FEATURES = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
]
DIABETES_TARGET = 'Outcome'
zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
diabetes_model = diabetes.drop_duplicates().copy()
diabetes_model[zero_as_missing] = diabetes_model[zero_as_missing].replace(0, np.nan)
X_diabetes = diabetes_model[DIABETES_FEATURES]
y_diabetes = diabetes_model[DIABETES_TARGET]
print('Feature vector dimension d =', len(DIABETES_FEATURES))

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1



Missing values:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,0,0,0,0,0,0,0,0,0



Duplicate rows: 0

Target distribution:


,count
Outcome,
0,500
1,268


Feature vector dimension d = 8


The diabetes dataset contains 768 observations and 8 input features. The target is `Outcome`. The model input is $X \in R^{N x 8}$ after imputation and standardization.

In [8]:
X_diabetes_train, X_diabetes_test, y_diabetes_train, y_diabetes_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.20, stratify=y_diabetes, random_state=RANDOM_STATE
)

diabetes_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

diabetes_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=19),
    'SVM RBF': SVC(probability=True, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1),
}

diabetes_pipelines = {}
diabetes_results = []
for name, estimator in diabetes_models.items():
    pipeline = Pipeline([('preprocessor', diabetes_preprocessor), ('model', estimator)])
    pipeline.fit(X_diabetes_train, y_diabetes_train)
    prediction = pipeline.predict(X_diabetes_test)
    diabetes_pipelines[name] = pipeline
    diabetes_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_diabetes_test, prediction),
        'Precision': precision_score(y_diabetes_test, prediction, zero_division=0),
        'Recall': recall_score(y_diabetes_test, prediction, zero_division=0),
        'F1': f1_score(y_diabetes_test, prediction, zero_division=0),
    })

diabetes_results = pd.DataFrame(diabetes_results).set_index('Model').sort_values('F1', ascending=False)
display(diabetes_results.round(4))

,Accuracy,Precision,Recall,F1
Model,,,,
Random Forest,0.8636,0.8235,0.7778,0.8000
SVM RBF,0.8377,0.7636,0.7778,0.7706
KNN,0.7857,0.7059,0.6667,0.6857
Logistic Regression,0.7078,0.5882,0.5556,0.5714


### Diabetes evaluation and controlled experiment

The primary comparison reports accuracy, precision, recall, and F1. Recall is especially important because missing a positive diabetes case can be costly. The controlled experiment below tests the tutorial’s question: **does feature scaling change KNN performance?**

In [9]:
diabetes_best_name = diabetes_results.index[0]
diabetes_best_pipeline = diabetes_pipelines[diabetes_best_name]
diabetes_best_prediction = diabetes_best_pipeline.predict(X_diabetes_test)
print('Selected model by F1:', diabetes_best_name)
print('Confusion matrix:')
print(confusion_matrix(y_diabetes_test, diabetes_best_prediction))

# Controlled experiment: scaled versus unscaled KNN.
knn_scaled = diabetes_pipelines['KNN']
knn_unscaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', KNeighborsClassifier(n_neighbors=19)),
])
knn_unscaled.fit(X_diabetes_train, y_diabetes_train)
experiment = pd.DataFrame({
    'Representation': ['Scaled features', 'Unscaled features'],
    'Accuracy': [
        accuracy_score(y_diabetes_test, knn_scaled.predict(X_diabetes_test)),
        accuracy_score(y_diabetes_test, knn_unscaled.predict(X_diabetes_test)),
    ],
})
display(experiment.round(4))

with open('diabetes_model.sav', 'wb') as file:
    pickle.dump(diabetes_best_pipeline, file)
print('Saved diabetes_model.sav')

Selected model by F1: Random Forest
Confusion matrix:
[[91  9]
 [12 42]]


,Representation,Accuracy
0,Scaled features,0.7857
1,Unscaled features,0.8506


Saved diabetes_model.sav


In [10]:
diabetes_sample = X_diabetes_test.iloc[[0]]
diabetes_prediction = int(diabetes_best_pipeline.predict(diabetes_sample)[0])
diabetes_probability = None
if hasattr(diabetes_best_pipeline, 'predict_proba'):
    diabetes_probability = float(diabetes_best_pipeline.predict_proba(diabetes_sample).max())
print('Sample input:')
display(diabetes_sample)
print('Prediction:', 'diabetic' if diabetes_prediction else 'non-diabetic')
print('Confidence:', None if diabetes_probability is None else round(diabetes_probability, 4))

Sample input:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
44,7,159,64.0,27,102.5,27.4,0.294,40


Prediction: non-diabetic
Confidence: 0.844
